In [1]:
import os
from pydantic_ai import Embedder
import chromadb
import base64
import pymupdf
from dotenv import load_dotenv
import openai
from markitdown import MarkItDown
import io

load_dotenv()

CHROMA_COLLECTION = "pdf_vectors"

In [2]:
embedder = Embedder("openai:text-embedding-3-small")

client = chromadb.PersistentClient(path="../chroma.db")

collection = client.get_or_create_collection(CHROMA_COLLECTION)

md = MarkItDown()

In [3]:
async def convert_and_embed_pdf(pdf_path: str):
    try:
        # open pdf file
        doc = pymupdf.open(pdf_path)

        if len(doc) == 0:
            raise ValueError("No pages found in PDF")
        
        single_page_doc = pymupdf.open()
        single_page_doc.insert_pdf(doc, from_page=0, to_page=1)

        pdf_bytes = single_page_doc.tobytes()
        single_page_doc.close()
        doc.close()

        page_md = md.convert(io.BytesIO(pdf_bytes), file_extension=".pdf")
        
        result = await embedder.embed_documents(page_md.text_content)

        emb = result.embeddings[0]

        collection.add(
            ids=[pdf_path],
            embeddings=[emb],
            metadatas = {
                "pdf_path": pdf_path,
                "category": pdf_path.split("/")[-2]
            }
        )
    except Exception as e:
        print(f"Error embedding {pdf_path}: {e}")
        return None


In [4]:
from tqdm.auto import tqdm

async def walk_and_embed(root_dir: str):
    pdfs = [
        os.path.join(root, file)
        for root, _, files in os.walk(root_dir)
        for file in files
        if file.endswith(".pdf")
    ]
    for pdf_path in tqdm(pdfs, desc=os.path.basename(root_dir)):
        await convert_and_embed_pdf(pdf_path)

/home/marco/development/pydantic-ai-document-processing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
for category in ["invoice", "packing_slip", "purchase_order", "quote", "shipping_order", "work_order"]:
    await walk_and_embed(f"../data/company-document-dataset-v2/test/{category}")

invoice:   0%|          | 0/6370 [00:00<?, ?it/s]

work_order: 100%|██████████| 5691/5691 [26:36<00:00,  3.56it/s]   
